In [1]:
# Import thư viện
import os
import glob
import shutil
import random
from pathlib import Path

In [2]:
# Input và Output
DATA_DIR   = "Dataset 1/images/images"
OUTPUT_DIR = "Dataset 1_organized"

output_dir = Path(OUTPUT_DIR)

if output_dir.exists():
    shutil.rmtree(output_dir)

# Chia tỉ lệ train/test
TRAIN_RATIO = 0.8   # 80% train, 20% test

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [3]:
VALID_EXTS = {'.png', '.jpg', '.jpeg'}

# Thu thập toàn bộ ảnh của class
def collect_images(cls_dir: Path) -> list[tuple[str, Path]]:
    images = []
    
    # Lấy ảnh trong default 
    for img_path in cls_dir.glob("default/*"):
        if img_path.is_file() and img_path.suffix.lower() in VALID_EXTS:
            images.append(("default", img_path))

    # Lấy ảnh trong real_world       
    for img_path in cls_dir.glob("real_world/*"):
        if img_path.is_file() and img_path.suffix.lower() in VALID_EXTS:
            images.append(("real", img_path))
 
    return images

In [4]:
# Đổi tên ảnh để tránh trùng lặp. 
# Ví dụ: aerosol_cans_default_img001.png
def build_dest_name(cls: str, domain: str, original_name: str) -> str:
    return f"{cls}_{domain}_{original_name}"

In [5]:

# Số class trong dataset 1
def organize_dataset(data_dir: str, output_dir: str, train_ratio: float = 0.8):
    data_dir   = Path(data_dir)
    output_dir = Path(output_dir)
 
    classes = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
 
    if not classes:
        raise ValueError(f"Không tìm thấy class nào trong: {data_dir}")
 
    print(f"Tìm thấy {len(classes)} class\n")

    total_train = 0
    total_test  = 0
 
    for cls in classes:
        cls_dir = data_dir / cls
        images  = collect_images(cls_dir)  # list of (domain, path)
 
        if not images:
            print(f"  [CẢNH BÁO] Class '{cls}' không có ảnh, bỏ qua.")
            continue
        # Xáo trộn trước khi chia
        random.shuffle(images)
 
        split_idx  = int(len(images) * train_ratio)
        train_imgs = images[:split_idx]
        test_imgs  = images[split_idx:]
 
        # Tạo thư mục đích
        train_cls_dir = output_dir / "train" / cls
        test_cls_dir  = output_dir / "test"  / cls
        train_cls_dir.mkdir(parents=True, exist_ok=True)
        test_cls_dir.mkdir(parents=True, exist_ok=True)
 
        for domain, src_path in train_imgs:
            dest_name = build_dest_name(cls, domain, src_path.name)
            shutil.copy2(src_path, train_cls_dir / dest_name)
 
        for domain, src_path in test_imgs:
            dest_name = build_dest_name(cls, domain, src_path.name)
            shutil.copy2(src_path, test_cls_dir / dest_name)
 
        total_train += len(train_imgs)
        total_test  += len(test_imgs)
 
        print(f"  [{cls}]  Tổng số ảnh trong class={len(images):4d}  |  Số ảnh train={len(train_imgs):4d}  |  Số ảnh test={len(test_imgs):4d}")
 
    print("TỔNG KẾT DATASET")
    print(f"Train images : {total_train}")
    print(f"Test images  : {total_test}")
    print(f"Total images : {total_train + total_test}")
    print(f"Saved to     : {output_dir}")

In [6]:
if __name__ == "__main__":
    organize_dataset(DATA_DIR, OUTPUT_DIR, TRAIN_RATIO)

Tìm thấy 30 class

  [aerosol_cans]  Tổng số ảnh trong class= 500  |  Số ảnh train= 400  |  Số ảnh test= 100
  [aluminum_food_cans]  Tổng số ảnh trong class= 500  |  Số ảnh train= 400  |  Số ảnh test= 100
  [aluminum_soda_cans]  Tổng số ảnh trong class= 500  |  Số ảnh train= 400  |  Số ảnh test= 100
  [cardboard_boxes]  Tổng số ảnh trong class= 500  |  Số ảnh train= 400  |  Số ảnh test= 100
  [cardboard_packaging]  Tổng số ảnh trong class= 500  |  Số ảnh train= 400  |  Số ảnh test= 100
  [clothing]  Tổng số ảnh trong class= 500  |  Số ảnh train= 400  |  Số ảnh test= 100
  [coffee_grounds]  Tổng số ảnh trong class= 500  |  Số ảnh train= 400  |  Số ảnh test= 100
  [disposable_plastic_cutlery]  Tổng số ảnh trong class= 500  |  Số ảnh train= 400  |  Số ảnh test= 100
  [eggshells]  Tổng số ảnh trong class= 500  |  Số ảnh train= 400  |  Số ảnh test= 100
  [food_waste]  Tổng số ảnh trong class= 500  |  Số ảnh train= 400  |  Số ảnh test= 100
  [glass_beverage_bottles]  Tổng số ảnh trong class=